# 📊 SIR Model Predictions vs Ground Truth

In this notebook, we visualize how well our trained machine learning model predicts the mean behavior of the SIR (Susceptible-Infected-Recovered) model from stochastic simulations.

In [ ]:
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import numpy as np

# Load data
df = pd.read_csv("../data/processed/sir_mean.csv")
X = df[["beta", "gamma", "time"]].values
y = df[["S", "I", "R"]].values

# Scale features and outputs
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y)
X_tensor = torch.tensor(X_scaled, dtype=torch.float32)

# Define model
import torch.nn as nn
class SIRNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 3)
        )
    def forward(self, x):
        return self.net(x)

# Load trained weights
model = SIRNet()
model.load_state_dict(torch.load("../models/sir_mlp.pt"))
model.eval()

# Predictions
y_pred_scaled = model(X_tensor).detach().numpy()
y_pred = scaler_y.inverse_transform(y_pred_scaled)

# R² scores
r2_S = r2_score(y[:, 0], y_pred[:, 0])
r2_I = r2_score(y[:, 1], y_pred[:, 1])
r2_R = r2_score(y[:, 2], y_pred[:, 2])
print(f"R² scores:\nS: {r2_S:.4f}\nI: {r2_I:.4f}\nR: {r2_R:.4f}")

In [ ]:
# Plotting true vs predicted for a few β-γ pairs
unique_pairs = df.groupby(['beta', 'gamma']).groups
pairs_to_plot = list(unique_pairs.items())[:3]  # You can increase this if needed

for (beta, gamma), indices in pairs_to_plot:
    idx = list(indices)
    time = df.iloc[idx]["time"].values
    true_vals = y[idx]
    pred_vals = y_pred[idx]

    plt.figure(figsize=(14, 4))
    for i, label in enumerate(["S", "I", "R"]):
        plt.subplot(1, 3, i+1)
        plt.plot(time, true_vals[:, i], label="True", linewidth=2)
        plt.plot(time, pred_vals[:, i], label="Pred", linestyle="--", linewidth=2)
        plt.title(f"{label}(t)", fontsize=12)
        plt.xlabel("Time")
        plt.ylabel(label)
        plt.legend()
    plt.suptitle(f"β = {beta}, γ = {gamma}", fontsize=14)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()